<a href="https://colab.research.google.com/github/castaneda-a/gesta/blob/main/gesta_tutorial.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Gesta — Tutorial

**Gesta** is a Python library for business administration — appointments, transactions, payments, and reporting in a clean, extensible API.

This notebook walks you through the full workflow of Gesta using `WellnessStudio`, a ready-to-use extension for integral wellness businesses (massage, yoga, therapy, etc.).

---

### What we'll cover

1. Installation
2. Initializing the studio
3. Registering clients and providers
4. Booking appointments
5. Registering transactions
6. Registering payments
7. Selling products directly
8. Reports and metrics
9. Handling errors
10. Extending Gesta for your own business

---
## 1. Installation

In [20]:
#%pip install gesta --quiet

In [21]:
import gesta
print(f"Gesta version: {gesta.__version__}")

Gesta version: 0.2.0


---
## 2. Initializing the studio

`WellnessStudio` is a pre-built extension of `Gesta` configured for wellness businesses.

Calling `setup()` creates the database tables and loads sample services and products.
We use `sqlite:///:memory:` — an in-memory database that resets when the notebook restarts, perfect for tutorials.

In [22]:
from gesta.extensions import WellnessStudio

studio = WellnessStudio(db_url="sqlite:///:memory:")
studio.setup(load_sample_data=True)

print("Studio initialized.")
print(f"Database reachable: {studio.ping()}")

Studio initialized.
Database reachable: True


### Pre-loaded sample data

`setup()` created the following for you:

| ID | Name | Price | Cost | Duration |
|---|---|---|---|---|
| `svc_masaje_sueco` | Swedish massage | $600 | $150 | 60 min |
| `svc_masaje_piedras` | Hot stone massage | $800 | $200 | 75 min |
| `svc_yoga_grupal` | Group yoga class | $120/person | $300 | 60 min |
| `svc_meditacion` | Meditation session | $100/person | $200 | 45 min |
| `svc_terapia` | Holistic therapy | $900 | $250 | 90 min |
| `prd_aceite_lavanda` | Lavender essential oil | $180 | $60 | — |
| `prd_incienso` | Sandalwood incense | $80 | $25 | — |
| `prd_vela` | Aromatic candle | $220 | $70 | — |
| `prd_hierbas` | Relaxing herb mix | $120 | $30 | — |

---
## 3. Registering clients and providers

All database operations happen inside a `with studio.session() as s:` block.
The session automatically commits on success and rolls back on any exception.

> **Important:** Any access to relationships (`service`, `clients`, `providers`) must happen **inside** the `with` block. Once the block closes the session ends and lazy loading is no longer available. Simple column attributes like `.id`, `.amount`, and `.status` are safe to use outside the block.

In [23]:
# Register clients — save IDs for use in later cells
with studio.session() as s:
    ana   = studio.add_client(s, name="Ana García",   phone="5551234567", email="ana@mail.com")
    luis  = studio.add_client(s, name="Luis Ramos",   phone="5559876543", email="luis@mail.com")
    sofia = studio.add_client(s, name="Sofía Torres", email="sofia@mail.com")

    ana_id   = ana.id
    luis_id  = luis.id
    sofia_id = sofia.id

    print("Clients registered:")
    print(f"  Ana   — {ana_id[:8]}...")
    print(f"  Luis  — {luis_id[:8]}...")
    print(f"  Sofía — {sofia_id[:8]}...")

Clients registered:
  Ana   — 9073de61...
  Luis  — cc12ae23...
  Sofía — e2306fa5...


In [24]:
# Register providers
with studio.session() as s:
    marta = studio.add_provider(s, name="Marta López", role="therapist",  email="marta@mail.com")
    pedro = studio.add_provider(s, name="Pedro Soto",  role="instructor", email="pedro@mail.com")

    marta_id = marta.id
    pedro_id = pedro.id

    print("Providers registered:")
    print(f"  Marta (therapist)  — {marta_id[:8]}...")
    print(f"  Pedro (instructor) — {pedro_id[:8]}...")

Providers registered:
  Marta (therapist)  — b857d750...
  Pedro (instructor) — a2d7a8f4...


In [25]:
# List all clients and providers
# Relationships (roles) must be accessed inside the session
with studio.session() as s:
    clients   = studio.list_clients(s)
    providers = studio.list_providers(s)

    print("Clients:")
    for c in clients:
        print(f"  {c.name} — {c.email}")

    print("\nProviders:")
    for p in providers:
        roles = ", ".join(r.name for r in p.roles)
        print(f"  {p.name} — roles: {roles}")

Clients:
  Ana García — ana@mail.com
  Luis Ramos — luis@mail.com
  Sofía Torres — sofia@mail.com

Providers:
  Marta López — roles: therapist
  Pedro Soto — roles: instructor


---
## 4. Booking appointments

An `Appointment` is a reservation for a future service. It captures:
- **who** is attending (clients)
- **who** is delivering (providers)
- **what** service
- **when** it will happen

Gesta automatically validates that the scheduled time is in the future and that no provider has a conflicting appointment.

In [26]:
from datetime import datetime, timedelta

next_monday = datetime.now() + timedelta(days=7)

# Book a Swedish massage for Ana with Marta
# Access relationships (service, clients, providers) inside the session
with studio.session() as s:
    appt1 = studio.appointments(s).book(
        service_id   = "svc_masaje_sueco",
        client_ids   = [ana_id],
        provider_ids = [marta_id],
        scheduled_at = next_monday.replace(hour=10, minute=0, second=0),
        notes        = "Client prefers light pressure",
    )
    appt1_id = appt1.id

    print(f"Booked: {appt1.service.name}")
    print(f"  When:     {appt1.scheduled_at.strftime('%A %b %d at %H:%M')}")
    print(f"  Clients:  {[c.name for c in [p for p in appt1.persons if p.is_recipient]]}")
    print(f"  Provider: {[p.name for p in [p for p in appt1.persons if p.is_provider]]}")
    print(f"  Status:   {appt1.status.value}")

Booked: Masaje sueco
  When:     Saturday May 23 at 10:00
  Clients:  ['Ana García']
  Provider: ['Marta López']
  Status:   scheduled


In [27]:
# Book a group yoga class for Ana, Luis, and Sofía with Pedro
# Price is $120 per person — total will be $120 × 3 = $360
with studio.session() as s:
    appt2 = studio.appointments(s).book(
        service_id   = "svc_yoga_grupal",
        client_ids   = [ana_id, luis_id, sofia_id],
        provider_ids = [pedro_id],
        scheduled_at = next_monday.replace(hour=12, minute=0, second=0),
    )
    appt2_id = appt2.id

    print(f"Booked: {appt2.service.name}")
    print(f"  Clients:          {[c.name for c in [p for p in appt2.persons if p.is_recipient]]}")
    print(f"  Price per person: ${appt2.service.price}")

Booked: Clase de yoga grupal
  Clients:          ['Ana García', 'Luis Ramos', 'Sofía Torres']
  Price per person: $120.00


In [28]:
# List all upcoming appointments
with studio.session() as s:
    upcoming = studio.appointments(s).list_upcoming()

    print(f"Upcoming appointments ({len(upcoming)}):")
    for appt in upcoming:
        clients = ", ".join(c.name for c in [p for p in appt.persons if p.is_recipient])
        print(f"  [{appt.scheduled_at.strftime('%H:%M')}] {appt.service.name} — {clients}")

Upcoming appointments (2):
  [10:00] Masaje sueco — Ana García
  [12:00] Clase de yoga grupal — Ana García, Luis Ramos, Sofía Torres


### Rescheduling an appointment

In [29]:
# Reschedule Ana's massage to 11am instead of 10am
with studio.session() as s:
    appt = studio.appointments(s).reschedule(
        appointment_id = appt1_id,
        new_datetime   = next_monday.replace(hour=11, minute=0, second=0),
    )
    print(f"Rescheduled to: {appt.scheduled_at.strftime('%H:%M')}")

Rescheduled to: 11:00


---
## 5. Registering transactions

A `Transaction` is a record that a service was delivered or a product was sold. It is distinct from an `Appointment`:

- An `Appointment` is a **future intention**
- A `Transaction` is a **past fact**

The most common flow is `register_from_appointment` — it converts a completed appointment into a transaction and automatically marks the appointment as `COMPLETED`.

Each transaction tracks:
- `amount` — total charged (`price × clients`)
- `cost_amount` — actual cost to the business
- `profit` — `amount - cost_amount`

In [30]:
# Register the transaction for Ana's massage
with studio.session() as s:
    tx1 = studio.transactions(s).register_from_appointment(
        appointment_id = appt1_id,
        occurred_at    = datetime.now(),
    )
    tx1_id = tx1.id

    print(f"Transaction: {tx1.service.name}")
    print(f"  Amount:   ${tx1.amount}")
    print(f"  Cost:     ${tx1.cost_amount}")
    print(f"  Profit:   ${tx1.profit}")
    print(f"  Status:   {tx1.status.value}")

Transaction: Masaje sueco
  Amount:   $600.00
  Cost:     $150.00
  Profit:   $450.00
  Status:   pending


In [31]:
# Register the group yoga transaction
# Amount = $120/person × 3 clients = $360
# Cost   = $300 flat (1 class regardless of how many attend)
with studio.session() as s:
    tx2 = studio.transactions(s).register_from_appointment(
        appointment_id = appt2_id,
        occurred_at    = datetime.now(),
    )
    tx2_id = tx2.id

    print(f"Transaction: {tx2.service.name}")
    print(f"  Amount:           ${tx2.amount}")
    print(f"  Cost:             ${tx2.cost_amount}")
    print(f"  Profit:           ${tx2.profit}")

Transaction: Clase de yoga grupal
  Amount:           $360.00
  Cost:             $900.00
  Profit:           $-540.00


In [32]:
# Applying a discount by overriding the amount
from decimal import Decimal

# Book a new appointment first
with studio.session() as s:
    appt3 = studio.appointments(s).book(
        service_id   = "svc_masaje_sueco",
        client_ids   = [luis_id],
        provider_ids = [marta_id],
        scheduled_at = next_monday.replace(hour=14, minute=0, second=0),
    )
    appt3_id = appt3.id
    print(f"Booked appointment for Luis: {appt3_id[:8]}...")

# Register with 20% discount
with studio.session() as s:
    tx3 = studio.transactions(s).register_from_appointment(
        appointment_id = appt3_id,
        occurred_at    = datetime.now(),
        amount         = Decimal("480.00"),  # $600 - 20% discount
        notes          = "20% loyalty discount applied",
    )
    tx3_id     = tx3.id
    orig_price = tx3.service.price

    print(f"Discounted transaction: ${tx3.amount} (original price: ${orig_price})")

Booked appointment for Luis: 4601f1cc...
Discounted transaction: $480.00 (original price: $600.00)


---
## 6. Registering payments

A `Payment` records money received. Key features:
- One payment can cover **multiple transactions** at once
- Payments can be **partial** — the transaction stays `PARTIAL` until fully paid
- Transaction status updates **automatically**: `PENDING` → `PARTIAL` → `PAID`

In [33]:
from gesta.core.entities import PaymentMethod

# Ana pays for her massage in cash
with studio.session() as s:
    studio.payments(s).register(
        transaction_ids = [tx1_id],
        amount          = Decimal("600.00"),
        method          = PaymentMethod.CASH,
        paid_at         = datetime.now(),
    )

# Check transaction status after payment
with studio.session() as s:
    tx = studio.transactions(s).get(tx1_id)
    print(f"Ana's massage after payment:")
    print(f"  Status:  {tx.status.value}")
    print(f"  Paid:    ${tx.amount_paid}")
    print(f"  Balance: ${tx.balance}")

Ana's massage after payment:
  Status:  paid
  Paid:    $600.00
  Balance: $0.00


In [34]:
# Partial payment for the yoga class
with studio.session() as s:
    studio.payments(s).register(
        transaction_ids = [tx2_id],
        amount          = Decimal("200.00"),  # partial — $360 total
        method          = PaymentMethod.CARD,
        paid_at         = datetime.now(),
        notes           = "First installment",
    )

with studio.session() as s:
    tx = studio.transactions(s).get(tx2_id)
    print(f"Group yoga after partial payment:")
    print(f"  Status:  {tx.status.value}")
    print(f"  Paid:    ${tx.amount_paid}")
    print(f"  Balance: ${tx.balance}")

Group yoga after partial payment:
  Status:  partial
  Paid:    $200.00
  Balance: $160.00


In [35]:
# Pay the remaining balance
with studio.session() as s:
    tx    = studio.transactions(s).get(tx2_id)
    remaining = tx.balance

with studio.session() as s:
    studio.payments(s).register(
        transaction_ids = [tx2_id],
        amount          = remaining,
        method          = PaymentMethod.TRANSFER,
        paid_at         = datetime.now(),
        notes           = "Final payment",
    )

with studio.session() as s:
    tx = studio.transactions(s).get(tx2_id)
    print(f"Group yoga after full payment:")
    print(f"  Status:  {tx.status.value}")
    print(f"  Balance: ${tx.balance}")

Group yoga after full payment:
  Status:  paid
  Balance: $0.00


In [36]:
# Pay Luis's discounted massage
with studio.session() as s:
    studio.payments(s).register(
        transaction_ids = [tx3_id],
        amount          = Decimal("480.00"),
        method          = PaymentMethod.CARD,
        paid_at         = datetime.now(),
    )
    print("Luis's discounted massage paid.")

Luis's discounted massage paid.


### Refunds

In [37]:
# Book, complete, and pay a new transaction — then issue a refund
with studio.session() as s:
    appt_refund = studio.appointments(s).book(
        service_id   = "svc_terapia",
        client_ids   = [sofia_id],
        provider_ids = [marta_id],
        scheduled_at = datetime.now() + timedelta(days=14),
    )
    appt_refund_id = appt_refund.id

with studio.session() as s:
    tx_refund    = studio.transactions(s).register_from_appointment(
        appointment_id = appt_refund_id,
        occurred_at    = datetime.now(),
    )
    tx_refund_id = tx_refund.id

with studio.session() as s:
    studio.payments(s).register(
        transaction_ids = [tx_refund_id],
        amount          = Decimal("900.00"),
        method          = PaymentMethod.CARD,
        paid_at         = datetime.now(),
    )

# Issue the refund
with studio.session() as s:
    studio.payments(s).register_refund(
        transaction_ids = [tx_refund_id],
        amount          = Decimal("900.00"),
        method          = PaymentMethod.CARD,
        paid_at         = datetime.now(),
        notes           = "Client cancelled — full refund",
    )

with studio.session() as s:
    tx = studio.transactions(s).get(tx_refund_id)
    print(f"After refund — status: {tx.status.value}")

After refund — status: refunded


---
## 7. Reports and metrics

The `ReportManager` provides aggregated summaries of your business data.
All report data is accessed inside the session.

In [38]:
start = datetime.now().replace(day=1, hour=0, minute=0, second=0, microsecond=0)
end   = datetime.now()

with studio.session() as s:
    summary = studio.reports(s).monthly_summary(
        year  = datetime.now().year,
        month = datetime.now().month,
    )
    print("Monthly Revenue Summary")
    print(f"  Transactions:  {summary.transaction_count}")
    print(f"  Payments:      {summary.payment_count}")
    print(f"  Total revenue: ${summary.total_revenue}")
    print(f"  Total cost:    ${summary.total_cost}")
    print(f"  Total profit:  ${summary.total_profit}")
    print(f"  Profit margin: {summary.profit_margin}%")
    print(f"  Collected:     ${summary.total_collected}")
    print(f"  Pending:       ${summary.total_pending}")
    print(f"  By method:     {summary.by_method}")

Monthly Revenue Summary
  Transactions:  4
  Payments:      5
  Total revenue: $2340.00
  Total cost:    $1450.00
  Total profit:  $890.00
  Profit margin: 38.03%
  Collected:     $2340.00
  Pending:       $0.00
  By method:     {'cash': Decimal('600.00'), 'card': Decimal('1580.00'), 'transfer': Decimal('160.00')}


In [39]:
with studio.session() as s:
    top_offerings = studio.reports(s).most_popular_offerings(
        start = start,
        end   = end,
        top   = 5,
    )
    print("Most popular offerings:")
    for o in top_offerings:
        print(f"  {o.offering_name:<30}  {o.count}x  revenue: ${o.total_revenue}  margin: {o.profit_margin}%")

Most popular offerings:
  Masaje sueco                    2x  revenue: $1080.00  margin: 72.22%
  Clase de yoga grupal            1x  revenue: $360.00  margin: -150.0%
  Terapia holística               1x  revenue: $900.00  margin: 72.22%


In [40]:
with studio.session() as s:
    top_clients = studio.reports(s).most_frequent_clients(
        start = start,
        end   = end,
    )
    print("Most frequent clients:")
    for c in top_clients:
        print(f"  {c.person_name:<20}  {c.transaction_count} transactions  spent: ${c.total_spent:.2f}")

Most frequent clients:
  Ana García            2 transactions  spent: $390.00
  Luis Ramos            2 transactions  spent: $330.00
  Sofía Torres          2 transactions  spent: $540.00


In [41]:
with studio.session() as s:
    top_providers = studio.reports(s).most_active_providers(
        start = start,
        end   = end,
    )
    print("Most active providers:")
    for p in top_providers:
        print(f"  {p.person_name:<20}  {p.transaction_count} sessions  generated: ${p.total_generated}")

Most active providers:
  Marta López           3 sessions  generated: $1980.00
  Pedro Soto            1 sessions  generated: $360.00


In [42]:
with studio.session() as s:
    appt_summary = studio.reports(s).appointment_summary(
        start = start,
        end   = end + timedelta(days=30),
    )
    print("Appointment summary:")
    print(f"  Total:           {appt_summary.total}")
    print(f"  Completed:       {appt_summary.completed}")
    print(f"  Scheduled:       {appt_summary.scheduled}")
    print(f"  Cancelled:       {appt_summary.cancelled}")
    print(f"  No-shows:        {appt_summary.no_show}")
    print(f"  Completion rate: {appt_summary.completion_rate}%")

Appointment summary:
  Total:           4
  Completed:       4
  Scheduled:       0
  Cancelled:       0
  No-shows:        0
  Completion rate: 100.0%


In [43]:
import json

with studio.session() as s:
    rows = studio.reports(s).export_transactions_to_dict(
        start = start,
        end   = end,
    )
    print(f"Exported {len(rows)} transactions.")
    print("\nFirst transaction:")
    print(json.dumps(rows[0], indent=2))

Exported 4 transactions.

First transaction:
{
  "id": "a1a4c2b2-fb72-4c7c-9d80-c010ae36f879",
  "occurred_at": "2026-05-16T05:45:26.983993",
  "offering": "Masaje sueco",
  "clients": "Ana Garc\u00eda",
  "client_count": 1,
  "providers": "Marta L\u00f3pez",
  "amount": "600.00",
  "price_per_client": "300.00",
  "cost_amount": "150.00",
  "profit": "450.00",
  "amount_paid": "600.00",
  "balance": "0.00",
  "status": "paid"
}


---
## 8. Handling errors

Gesta raises descriptive exceptions that you can catch individually or as a group.
All exceptions inherit from `GestaError`.

In [44]:
from gesta import (
    ValidationError,
    AppointmentConflictError,
    NotFoundError,
    BusinessRuleError,
    GestaError,
)

# 1. Scheduling in the past
print("1. Scheduling in the past:")
try:
    with studio.session() as s:
        studio.appointments(s).book(
            service_id   = "svc_masaje_sueco",
            client_ids   = [ana_id],
            provider_ids = [marta_id],
            scheduled_at = datetime.now() - timedelta(days=1),
        )
except ValidationError as e:
    print(f"  ValidationError caught: {e}")

1. Scheduling in the past:
  ValidationError caught: El campo 'scheduled_at' debe ser una fecha y hora futura. Se recibió: 2026-05-15 05:45:27.178343


In [45]:
# 2. Schedule conflict — Marta already has a booking at 11am next Monday
print("2. Schedule conflict:")
try:
    with studio.session() as s:
        studio.appointments(s).book(
            service_id   = "svc_masaje_sueco",
            client_ids   = [sofia_id],
            provider_ids = [marta_id],
            scheduled_at = next_monday.replace(hour=11, minute=30, second=0),
        )
except AppointmentConflictError as e:
    print(f"  AppointmentConflictError caught: {e}")

2. Schedule conflict:


In [46]:
# 3. Entity not found
print("3. Entity not found:")
try:
    with studio.session() as s:
        studio.appointments(s).get("non-existent-id")
except NotFoundError as e:
    print(f"  NotFoundError caught: {e}")
    print(f"  Entity: {e.entity}, ID: {e.identifier}")

3. Entity not found:
  NotFoundError caught: Appointment no encontrado: 'non-existent-id'
  Entity: Appointment, ID: non-existent-id


In [47]:
# 4. Cancelling an appointment that has a transaction
print("4. Cancelling an appointment with a transaction:")
try:
    with studio.session() as s:
        studio.appointments(s).cancel(appt1_id)
except BusinessRuleError as e:
    print(f"  BusinessRuleError caught: {e}")

4. Cancelling an appointment with a transaction:
  BusinessRuleError caught: No se puede cancelar la cita '8f9a4be7-11ca-4f73-8e86-37a1d7a257dd' porque ya tiene una transacción asociada (id='a1a4c2b2-fb72-4c7c-9d80-c010ae36f879'). Gestiona la transacción directamente.


---
## Summary

You've seen the full Gesta workflow:

| Step | What happens |
|---|---|
| `studio.setup()` | Initialize database, roles, and sample data |
| `add_client / add_provider` | Register people with roles |
| `appointments.book()` | Create a future reservation |
| `appointments.reschedule()` | Change the date/time |
| `transactions.register_from_appointment()` | Deliver the service → creates transaction |
| `transactions.register()` | Direct sale without appointment |
| `payments.register()` | Record money received |
| `payments.register_refund()` | Issue a refund |
| `reports.monthly_summary()` | Revenue, cost, profit overview |
| `reports.most_popular_offerings()` | Top services and products |
| `reports.export_transactions_to_dict()` | Export data to CSV/JSON |

---

### Resources

- **PyPI**: https://pypi.org/project/gesta
- **GitHub**: https://github.com/castaneda-a/gesta-py
- **License**: MIT